# Chapter 02: Text Normalization and Unicode in NLP
> **Language AI Handbook — Comprehensive Interactive Study Notebook**  
> *Companion Guide & In-Depth Exploration based on Michael Brenndoerfer's Language AI Handbook*

---

### Executive Summary & Learning Roadmap
In Natural Language Processing (NLP), **Text Normalization** is the foundational stage where raw textual strings are converted into consistent, canonical representations. Before tokenization, embedding generation, or model training can occur, we must resolve a fundamental question: **When should two different byte or character sequences be considered computationally identical?**

In this comprehensive notebook, we cover:
1. **The Core Problem**: Why visual equality $\neq$ computational equality (the infamous *café* problem).
2. **Unicode Normalization Forms**: The $2 \times 2$ matrix of **NFC**, **NFD**, **NFKC**, and **NFKD**.
3. **Case Folding vs. Lowercasing**: Why `.lower()` breaks in multilingual corpora (German $\text{\ss}$, Greek $\sigma/\varsigma/\Sigma$, Turkish $I/\text{\u0131}/\dot{I}/i$).
4. **Accent & Diacritic Handling**: Lossless vs. lossy transformations, Unicode category `Mn` (Mark, Nonspacing), and semantic collision hazards (e.g., Spanish *año* vs. *ano*).
5. **Whitespace Normalization**: Taming non-breaking spaces, em/en spaces, ideographic spaces, and the invisible **Zero-Width Space** (`U+200B`) that breaks `.isspace()`.
6. **Ligatures & Digraphs**: Typographic expansion vs. linguistic preservation.
7. **Full-Width to Half-Width Conversion**: Handling East Asian typography and arithmetic code offsets.
8. **End-to-End Normalization Pipelines**: Building modular, production-ready `TextNormalizer` pipelines for both *Conservative Storage* and *Aggressive Search*.
9. **Practical Application**: Real-world entity deduplication.
10. **Tradeoffs, Homograph Attacks & Security**: Defending against malicious Unicode spoofing.
11. **Self-Assessment Quiz**: 10 interactive questions with immediate automated grading and explanatory feedback.

---
## 1. Why Normalization Matters: Visual Identity vs. Computational Equality

Consider a standard NLP search or classification task: a user queries `"café"`.
In Unicode, this word can be stored in at least two visually indistinguishable ways:
1. **Precomposed Form**: `c` + `a` + `f` + `é` (`U+00E9`: Latin Small Letter E with Acute)
2. **Decomposed Form**: `c` + `a` + `f` + `e` (`U+0065`) + `́` (`U+0301`: Combining Acute Accent)

To a human eye, both render identically as **café**. But to Python's standard string comparison (`==`), they are completely unequal!

Let's test this in code:

In [1]:
# Two visually identical strings
cafe_precomposed = "café"          # c, a, f, é (U+00E9)
cafe_decomposed = "café"          # c, a, f, e (U+0065) + combining acute (U+0301)

print(f"String 1: '{cafe_precomposed}'")
print(f"String 2: '{cafe_decomposed}'")
print("-" * 50)
print(f"Visually identical?              -> Yes ('café')")
print(f"Python string equality (==)?       -> {cafe_precomposed == cafe_decomposed}")
print(f"Lengths: cafe1={len(cafe_precomposed)} vs cafe2={len(cafe_decomposed)}")
print(f"UTF-8 Bytes cafe1: {list(cafe_precomposed.encode('utf-8'))}")
print(f"UTF-8 Bytes cafe2: {list(cafe_decomposed.encode('utf-8'))}")
print()
print("Code point breakdown:")
print(f"cafe1: {[f'U+{ord(c):04X} ({c})' for c in cafe_precomposed]}")
print(f"cafe2: {[f'U+{ord(c):04X} ({c})' for c in cafe_decomposed]}")

String 1: 'café'
String 2: 'café'
--------------------------------------------------
Visually identical?              -> Yes ('café')
Python string equality (==)?       -> False
Lengths: cafe1=4 vs cafe2=5
UTF-8 Bytes cafe1: [99, 97, 102, 195, 169]
UTF-8 Bytes cafe2: [99, 97, 102, 101, 204, 129]

Code point breakdown:
cafe1: ['U+0063 (c)', 'U+0061 (a)', 'U+0066 (f)', 'U+00E9 (é)']
cafe2: ['U+0063 (c)', 'U+0061 (a)', 'U+0066 (f)', 'U+0065 (e)', 'U+0301 (́)']


> [!WARNING]
> ### The Compounding Cost of Unnormalized Text in NLP
> If you don't normalize text at the very beginning of your pipeline, the discrepancy cascades through every downstream layer:
> 
> | NLP Stage | Failure Mode Without Normalization |
> | :--- | :--- |
> | **Search & Retrieval** | Queries for `"café"` return zero hits against decomposed documents. Recall drops significantly. |
> | **Deduplication** | Identical records are flagged as distinct, polluting your datasets with near-duplicates. |
> | **Tokenization** | Subword tokenizers (BPE / WordPiece) may split `U+0301` into a separate orphan token, altering sequence lengths. |
> | **Vocabulary Size** | The vocabulary explodes with duplicate representations for identical semantic tokens, diluting frequency counts. |
> | **Embeddings** | The model learns two separate embedding vectors for the same concept, degrading semantic density. |

---
## 2. Unicode Normalization Forms: The $2 \times 2$ Architectural Matrix

The Unicode Consortium (Standard Annex #15) standardized **four normalization forms** to eliminate representation ambiguity. These forms are structured across two orthogonal axes:

$$\begin{array}{c|c|c}
\text{\textbf{Equivalence \textbackslash{} Operation}} & \text{\textbf{Composition (Compose)}} & \text{\textbf{Decomposition (Decompose)}} \\
\hline
\text{\textbf{Canonical}} & \textbf{NFC} \text{ (Form C)} & \textbf{NFD} \text{ (Form D)} \\
\hline
\text{\textbf{Compatibility}} & \textbf{NFKC} \text{ (Form KC)} & \textbf{NFKD} \text{ (Form KD)} \\
\end{array}$$

### The Core Distinctions
1. **Canonical Equivalence ($C / D$)**:
   - Preserves both visual appearance and semantic identity.
   - Strictly lossless: characters that represent the exact same abstract symbol are equated (e.g., precomposed `é` vs. `e` + `́`).
2. **Compatibility Equivalence ($KC / KD$)**:
   - Aggressive / lossy formatting normalization: maps characters that represent the same underlying semantic concept even if they differ in formatting or typography.
   - Maps ligatures (`ﬁ` $\rightarrow$ `fi`), superscripts (`²` $\rightarrow$ `2`), circled numbers (`①` $\rightarrow$ `1`), Roman numerals (`Ⅳ` $\rightarrow$ `IV`), and full-width characters (`Ａ` $\rightarrow$ `A`).
3. **Compose ($C$)**:
   - Combines decomposed characters back into single precomposed code points where available (shortest representation).
4. **Decompose ($D$)**:
   - Separates characters into their base glyphs and combining marks (longest representation).

In [2]:
import unicodedata

# 1. Canonical Composition (NFC)
decomposed_word = "café"
nfc_composed = unicodedata.normalize("NFC", decomposed_word)

print("--- NFC (Canonical Composition) ---")
print(f"Original (decomposed): '{decomposed_word}' (length {len(decomposed_word)})")
print(f"After NFC:             '{nfc_composed}' (length {len(nfc_composed)})")
print(f"Matches precomposed 'café'? -> {nfc_composed == 'café'}")

# 2. Canonical Decomposition (NFD)
precomposed_word = "café"
nfd_decomposed = unicodedata.normalize("NFD", precomposed_word)

print("\n--- NFD (Canonical Decomposition) ---")
print(f"Original (composed):   '{precomposed_word}' (length {len(precomposed_word)})")
print(f"After NFD:             '{nfd_decomposed}' (length {len(nfd_decomposed)})")
print(f"NFD Code points:       {[f'U+{ord(c):04X}' for c in nfd_decomposed]}")

--- NFC (Canonical Composition) ---
Original (decomposed): 'café' (length 5)
After NFC:             'café' (length 4)
Matches precomposed 'café'? -> True

--- NFD (Canonical Decomposition) ---
Original (composed):   'café' (length 4)
After NFD:             'café' (length 5)
NFD Code points:       ['U+0063', 'U+0061', 'U+0066', 'U+0065', 'U+0301']


### Compatibility Normalization: NFKC and NFKD
The `"K"` forms apply compatibility decomposition before canonical composition/decomposition. 
Think of **NFKC** as the **"Normalize for Search"** standard: when a user types a Roman numeral, a circled number, a ligature copied from a PDF, or a superscript, NFKC transforms them all to standard ASCII characters.

In [3]:
# Compatibility test cases
test_cases = [
    ("ﬁ", "fi ligature"),
    ("①", "circled digit one"),
    ("Ⅳ", "roman numeral four"),
    ("ｈｅｌｌｏ", "full-width 'hello'"),
    ("²", "superscript two"),
    ("㎝", "km metric symbol (single glyph)"),
    ("2½", "half fraction symbol")
]

print(f"{'Input':<12} {'Description':<26} {'NFKC Result':<14} {'Length (Before -> After)'}")
print("-" * 72)
for raw_char, desc in test_cases:
    nfkc_res = unicodedata.normalize("NFKC", raw_char)
    print(f"{raw_char:<12} {desc:<26} {nfkc_res:<14} {len(raw_char)} -> {len(nfkc_res)}")

Input        Description                NFKC Result    Length (Before -> After)
------------------------------------------------------------------------
ﬁ            fi ligature                fi             1 -> 2
①            circled digit one          1              1 -> 1
Ⅳ            roman numeral four         IV             1 -> 2
ｈｅｌｌｏ        full-width 'hello'         hello          5 -> 5
²            superscript two            2              1 -> 1
㎝            km metric symbol (single glyph) cm             1 -> 2
2½           half fraction symbol       21⁄2           2 -> 4


### Comparing All Four Normalization Forms
Let's build a general inspection utility to trace how any arbitrary string behaves across all 4 normalization forms:

In [4]:
def compare_normalization_forms(text: str):
    """Inspect and compare a string across NFC, NFD, NFKC, and NFKD."""
    forms = ["NFC", "NFD", "NFKC", "NFKD"]
    print(f"INPUT: '{text}' (Length: {len(text)} code points)")
    print("=" * 70)
    for form in forms:
        norm = unicodedata.normalize(form, text)
        codepoints = " ".join(f"U+{ord(c):04X}" for c in norm)
        print(f"{form:<6} | Len: {len(norm):<2} | Text: '{norm}'")
        print(f"       | Code points: {codepoints}")
    print()

# Test with a complex real-world string containing ligatures, accents, and circled digits
compare_normalization_forms("ﬁnancial résumé ①")

INPUT: 'ﬁnancial résumé ①' (Length: 17 code points)
NFC    | Len: 17 | Text: 'ﬁnancial résumé ①'
       | Code points: U+FB01 U+006E U+0061 U+006E U+0063 U+0069 U+0061 U+006C U+0020 U+0072 U+00E9 U+0073 U+0075 U+006D U+00E9 U+0020 U+2460
NFD    | Len: 19 | Text: 'ﬁnancial résumé ①'
       | Code points: U+FB01 U+006E U+0061 U+006E U+0063 U+0069 U+0061 U+006C U+0020 U+0072 U+0065 U+0301 U+0073 U+0075 U+006D U+0065 U+0301 U+0020 U+2460
NFKC   | Len: 18 | Text: 'financial résumé 1'
       | Code points: U+0066 U+0069 U+006E U+0061 U+006E U+0063 U+0069 U+0061 U+006C U+0020 U+0072 U+00E9 U+0073 U+0075 U+006D U+00E9 U+0020 U+0031
NFKD   | Len: 20 | Text: 'financial résumé 1'
       | Code points: U+0066 U+0069 U+006E U+0061 U+006E U+0063 U+0069 U+0061 U+006C U+0020 U+0072 U+0065 U+0301 U+0073 U+0075 U+006D U+0065 U+0301 U+0020 U+0031



> [!TIP]
> ### Quick Decision Guide: Which Form Should You Choose?
> 
> | Use Case | Recommended Form | Rationale |
> | :--- | :--- | :--- |
> | **General Database Storage & Display** | **NFC** | Compact, visually preserves original author typography, standard web convention. |
> | **Full-Text Search & Deduplication** | **NFKC** | Collapses ligatures, full-width glyphs, and compatibility variants into standard characters. |
> | **Accent / Diacritic Stripping** | **NFD** | Decomposes base letters from accents so combining marks (`Mn`) can be filtered cleanly. |
> | **Security & User Authentication** | **NFKC** | Eliminates visual spoofing of usernames across compatible glyph sets. |

---
## 3. Case Folding vs. Lowercasing: Escaping the ASCII Mindset

Every NLP beginner is taught to call `.lower()` on strings for case-insensitive comparison. In multilingual NLP, **simple lowercasing is a trap**.

### 3.1 The German Eszett (ß) Anomaly
In German, the lowercase character `ß` (*eszett* or sharp s) represents a double 's'. Historically, German orthography had no uppercase form (until the 2008 addition of the capital `ẞ`), so `ß` uppercases to `"SS"`.
However, lowercasing `"SS"` gives `"ss"`, **not** `"ß"`. 

This breaks round-trip invariance: `text.upper().lower() != text`!


In [5]:
german_word = "straße"  # street in German

lowered = german_word.lower()
uppered = german_word.upper()
round_trip = uppered.lower()

print(f"Original:   '{german_word}' (length {len(german_word)})")
print(f"lower():    '{lowered}' (length {len(lowered)})")
print(f"upper():    '{uppered}' (length {len(uppered)})")
print(f"Round-trip: '{round_trip}' (length {len(round_trip)})")
print(f"Does original == round_trip? -> {german_word == round_trip} (Failure with .lower()!)")
print()

# Now compare with casefold()
words = ["Straße", "STRASSE", "straße", "strasse", "STRAßE"]
print(f"Input variations of German 'street': {words}")
print(f"Distinct using lower():    {len(set(w.lower() for w in words))} distinct values -> {set(w.lower() for w in words)}")
print(f"Distinct using casefold(): {len(set(w.casefold() for w in words))} distinct value  -> {set(w.casefold() for w in words)}")

Original:   'straße' (length 6)
lower():    'straße' (length 6)
upper():    'STRASSE' (length 7)
Round-trip: 'strasse' (length 7)
Does original == round_trip? -> False (Failure with .lower()!)

Input variations of German 'street': ['Straße', 'STRASSE', 'straße', 'strasse', 'STRAßE']
Distinct using lower():    2 distinct values -> {'straße', 'strasse'}
Distinct using casefold(): 1 distinct value  -> {'strasse'}


### 3.2 Greek Sigma (σ, ς, Σ)
In Greek, the letter sigma has two lowercase forms depending on position:
- `σ` (lowercase medial sigma, used inside a word)
- `ς` (lowercase final sigma, used at the end of a word)
- `Σ` (uppercase sigma)

Python's `.casefold()` maps both `σ` and `ς` to `σ`, unifying them for search!

### 3.3 The Turkish Dotted and Dotless 'I'
In Turkish, there are four forms of 'I':
- `İ` (uppercase dotted I) $\leftrightarrow$ `i` (`U+0069`, lowercase dotted i)
- `I` (`U+0049`, uppercase dotless I) $\leftrightarrow$ `ı` (`U+0131`, lowercase dotless i)

Standard Python `.lower()` follows English rules: `"I".lower()` produces `"i"`, which corrupts Turkish text! In Turkish, `"ISTANBUL".lower()` should produce `"ıstanbul"` (with dotless ı), but Python produces `"istanbul"` (with dotted i). For Turkish corpora, locale-aware libraries (like PyICU) must be employed.


In [6]:
# Greek Sigma Demonstration
greek_medial = "σ"  # medial sigma
greek_final = "ς"   # final sigma
greek_upper = "Σ"   # capital sigma

print("--- Greek Sigma Normalization ---")
print(f"Medial sigma: {greek_medial} -> casefold: '{greek_medial.casefold()}'")
print(f"Final sigma:  {greek_final} -> casefold: '{greek_final.casefold()}'")
print(f"Upper sigma:  {greek_upper} -> casefold: '{greek_upper.casefold()}'")
print(f"Are all unified by casefold? -> {greek_medial.casefold() == greek_final.casefold() == greek_upper.casefold()}")

print("\n--- Turkish I Inspection ---")
turkish_examples = [
    ("I", "English uppercase I"),
    ("i", "English lowercase i"),
    ("İ", "Turkish uppercase dotted I (U+0130)"),
    ("ı", "Turkish lowercase dotless i (U+0131)")
]
print(f"{'Char':<6} {'Description':<38} {'lower()':<8} {'casefold()':<10}")
print("-" * 65)
for char, desc in turkish_examples:
    print(f"'{char}'    {desc:<38} '{char.lower()}'      '{char.casefold()}'")

--- Greek Sigma Normalization ---
Medial sigma: σ -> casefold: 'σ'
Final sigma:  ς -> casefold: 'σ'
Upper sigma:  Σ -> casefold: 'σ'
Are all unified by casefold? -> True

--- Turkish I Inspection ---
Char   Description                            lower()  casefold()
-----------------------------------------------------------------
'I'    English uppercase I                    'i'      'i'
'i'    English lowercase i                    'i'      'i'
'İ'    Turkish uppercase dotted I (U+0130)    'i̇'      'i̇'
'ı'    Turkish lowercase dotless i (U+0131)   'ı'      'ı'


---
## 4. Accent and Diacritic Handling: Clean Stripping via NFD & `Mn` Filtering

In search engines, query matching often requires **accent folding** (e.g., users typing `"naive"` should find `"naïve"`, and `"muller"` should find `"Müller"`).

### The Canonical 3-Step Accent Removal Algorithm
Rather than building a brittle, incomplete character replacement dictionary (`replace('é', 'e')`), the standard Unicode technique is:
1. **Decompose to NFD**: Splits base letters from accents (e.g., `é` becomes `e` + `́`).
2. **Filter Category `'Mn'`**: Unicode General Category **Mark, Nonspacing (`Mn`)** matches all combining accents. We strip any character with category `Mn`.
3. **Recompose to NFC**: Cleanly re-assembles the remaining characters into canonical form.

In [7]:
def remove_diacritics(text: str) -> str:
    """Remove diacritical marks cleanly using NFD decomposition and Mn filtering."""
    # Step 1: Decompose characters into base + combining marks
    decomposed = unicodedata.normalize("NFD", text)
    
    # Step 2: Filter out all combining marks (Category 'Mn' = Mark, Nonspacing)
    filtered = "".join(char for char in decomposed if unicodedata.category(char) != "Mn")
    
    # Step 3: Recompose into canonical NFC
    return unicodedata.normalize("NFC", filtered)

# Test across various languages
multilingual_test = [
    "résumé",     # French
    "naïve",      # French/English
    "Ñoño",      # Spanish
    "Zürich",     # German
    "Ångström",    # Swedish
    "crème brûlée" # French
]

print(f"{'Original':<18} {'Stripped (ASCII-clean)':<18}")
print("-" * 40)
for w in multilingual_test:
    print(f"{w:<18} {remove_diacritics(w):<18}")

Original           Stripped (ASCII-clean)
----------------------------------------
résumé             resume            
naïve              naive             
Ñoño               Nono              
Zürich             Zurich            
Ångström           Angstrom          
crème brûlée       creme brulee      


> [!CAUTION]
> ### Preserving Semantic Distinctions: The Collision Danger!
> Stripping diacritics is a **lossy compression**. In many languages, accents are not decorative; they are phonemic and distinguish completely different words:
> 
> | Language | Word with Accent | Meaning | Stripped Word | Collided Meaning |
> | :--- | :--- | :--- | :--- | :--- |
> | **Spanish** | *año* | Year | *ano* | Anus |
> | **Spanish** | *sí* | Yes | *si* | If |
> | **French** | *où* | Where | *ou* | Or |
> | **German** | *schön* | Beautiful | *schon* | Already |
> | **German** | *für* | For | *fur* | Fur (English loan) |
> | **English** | *résumé* | CV / Document | *resume* | Continue |
> 
> **Architectural Recommendation**: Use **Dual Storage** in production. Store the exact, unstripped NFC text for display and language models (NER, Translation), and create a secondary stripped field specifically for full-text search indexing.

---
## 5. Whitespace Normalization: Taming Invisible Unicode Spaces

Whitespace in real-world scraped text (PDFs, HTML web pages, OCR) is full of non-standard space characters:
- `U+00A0` (No-Break Space, generated by browser `&nbsp;` and PDF line-wrap preservation)
- `U+2002` / `U+2003` (En / Em Space in typography)
- `U+2009` (Thin Space in mathematical expressions)
- `U+3000` (Ideographic Space in CJK typography)

### The Invisible Killer: Zero-Width Characters
Characters like `U+200B` (Zero-Width Space), `U+200C` (Zero-Width Non-Joiner), `U+200D` (Zero-Width Joiner), and `U+FEFF` (Byte Order Mark) are invisible on screen, **yet they consume 3 bytes in UTF-8**.

Worst of all: **Python's `str.isspace()` returns `False` for Zero-Width Space!**

In [8]:
whitespace_samples = [
    ("\u0020", "Space (ASCII)"),
    ("\u00a0", "No-Break Space"),
    ("\u2002", "En Space"),
    ("\u2003", "Em Space"),
    ("\u2009", "Thin Space"),
    ("\u200b", "Zero Width Space (ZWSP)"),
    ("\u3000", "Ideographic Space (CJK)"),
    ("\t",     "Tab"),
    ("\n",     "Newline"),
]

print(f"{'Code Point':<12} {'Name':<28} {'UTF-8 Bytes':<14} {'isspace()?'}")
print("-" * 65)
for ch, name in whitespace_samples:
    hex_code = f"U+{ord(ch):04X}"
    byte_len = len(ch.encode('utf-8'))
    print(f"{hex_code:<12} {name:<28} {byte_len:<14} {ch.isspace()}")

Code Point   Name                         UTF-8 Bytes    isspace()?
-----------------------------------------------------------------
U+0020       Space (ASCII)                1              True
U+00A0       No-Break Space               2              True
U+2002       En Space                     3              True
U+2003       Em Space                     3              True
U+2009       Thin Space                   3              True
U+200B       Zero Width Space (ZWSP)      3              False
U+3000       Ideographic Space (CJK)      3              True
U+0009       Tab                          1              True
U+000A       Newline                      1              True


Let's write a battle-tested whitespace normalizer using regular expressions:

In [9]:
import re

def normalize_whitespace(text: str, collapse: bool = True, strip: bool = True) -> str:
    """Standardize all Unicode spaces, strip zero-width noise, and normalize line endings."""
    # 1. Map all Unicode space separator category (Zs) characters to standard ASCII space
    # Includes: U+0020, U+00A0, U+1680, U+2000-U+200A, U+202F, U+205F, U+3000
    whitespace_pattern = r"[\u0020\u00A0\u1680\u2000-\u200A\u202F\u205F\u3000]"
    text = re.sub(whitespace_pattern, " ", text)
    
    # 2. Strip invisible zero-width characters (ZWSP, ZWNJ, ZWJ, BOM)
    text = re.sub(r"[\u200B\u200C\u200D\uFEFF]", "", text)
    
    # 3. Standardize Windows (\r\n) and legacy Mac (\r) line endings to POSIX (\n)
    text = re.sub(r"\r\n|\r", "\n", text)
    
    if collapse:
        # Collapse multiple horizontal spaces to single space
        text = re.sub(r" +", " ", text)
        # Collapse excessive blank lines (3+ newlines to double newline)
        text = re.sub(r"\n{3,}", "\n\n", text)
        
    if strip:
        text = text.strip()
        
    return text

# Test with messy input containing NBSP, Zero-Width Space, and Ideographic space
messy_text = "Hello\u00a0\u00a0World\u200b!\u3000\u3000NLP\tTest\n\n\n\nDone."
cleaned_text = normalize_whitespace(messy_text)

print(f"Messy length:   {len(messy_text)} code points")
print(f"Cleaned length: {len(cleaned_text)} code points")
print("Cleaned text output:")
print(repr(cleaned_text))

Messy length:   33 code points
Cleaned length: 28 code points
Cleaned text output:
'Hello World! NLP\tTest\n\nDone.'


---
## 6. Ligature Expansion: Typographic vs. Linguistic

In printing and typography, adjacent characters often merge into single glyphs:
- **Typographic Ligatures** (Alphabetic Presentation Forms: `U+FB00`–`U+FB4F`):
  - `ﬁ` (fi), `ﬂ` (fl), `ﬀ` (ff), `ﬃ` (ffi), `ﬄ` (ffl), `ﬅ` (st).
  - Used solely for font aesthetics (e.g., stopping the dot of 'i' from hitting the hood of 'f').
- **Linguistic Digraphs**:
  - `æ` (ae), `œ` (oe), `Æ` (AE), `Œ` (OE).
  - These are legitimate independent letters in French, Danish, Norwegian, and Old English.

**Crucial Insight**: NFKC decomposes **typographic ligatures** (`ﬁ` $\rightarrow$ `fi`), but deliberately **preserves linguistic digraphs** (`æ` remains `æ`).

In [10]:
ligature_test = [
    ("ﬁ", "fi", "Typographic ligature 'fi'"),
    ("ﬂ", "fl", "Typographic ligature 'fl'"),
    ("ﬀ", "ff", "Typographic ligature 'ff'"),
    ("ﬃ", "ffi", "Typographic ligature 'ffi'"),
    ("œ", "oe", "Linguistic digraph 'oe' (French)"),
    ("æ", "ae", "Linguistic letter 'ae' (Danish/Norwegian)"),
]

print(f"{'Glyph':<6} {'Code Point':<12} {'NFKC Result':<12} {'Status'}")
print("-" * 55)
for glyph, expected, desc in ligature_test:
    nfkc_form = unicodedata.normalize("NFKC", glyph)
    status = "Expanded to ASCII" if nfkc_form == expected else "Preserved as Letter"
    print(f"'{glyph}'    U+{ord(glyph):04X}      '{nfkc_form}'         {status}")

Glyph  Code Point   NFKC Result  Status
-------------------------------------------------------
'ﬁ'    U+FB01      'fi'         Expanded to ASCII
'ﬂ'    U+FB02      'fl'         Expanded to ASCII
'ﬀ'    U+FB00      'ff'         Expanded to ASCII
'ﬃ'    U+FB03      'ffi'         Expanded to ASCII
'œ'    U+0153      'œ'         Preserved as Letter
'æ'    U+00E6      'æ'         Preserved as Letter


---
## 7. Full-Width to Half-Width Conversion (East Asian CJK Text)

East Asian text systems (Chinese, Japanese, Korean) include **full-width** versions of ASCII characters (`U+FF01` to `U+FF5E`) designed to occupy the exact square grid (em-square) of ideographs.

For instance, a Japanese corporate record might list `"ＩＢＭ"` (`U+FF29 U+FF22 U+FF2D`) rather than standard `"IBM"`.

### The Mathematical Mapping
Except for the ideographic space (`U+3000`), every full-width character maps to its standard ASCII equivalent by subtracting a constant arithmetic offset:
$$\text{ASCII Code} = \text{FullWidth Code} - \text{0xFEE0}$$

While NFKC handles this automatically, understanding the mapping reveals how Unicode structures compatibility blocks.

In [11]:
def fullwidth_to_halfwidth_manual(text: str) -> str:
    """Manual arithmetic conversion from full-width CJK to half-width ASCII."""
    result = []
    for char in text:
        code = ord(char)
        # Full-width ASCII block: U+FF01 ('！') to U+FF5E ('～')
        if 0xFF01 <= code <= 0xFF5E:
            result.append(chr(code - 0xFEE0))
        # Ideographic space: U+3000 -> ASCII space U+0020
        elif code == 0x3000:
            result.append(" ")
        else:
            result.append(char)
    return "".join(result)

fullwidth_sample = "Ｈｅｌｌｏ　Ｗｏｒｌｄ！　１２３"
manual_converted = fullwidth_to_halfwidth_manual(fullwidth_sample)
nfkc_converted = unicodedata.normalize("NFKC", fullwidth_sample)

print(f"Original Full-Width: {fullwidth_sample}")
print(f"Manual Half-Width:   {manual_converted}")
print(f"NFKC Half-Width:     {nfkc_converted}")
print(f"Do Manual and NFKC match exactly? -> {manual_converted == nfkc_converted}")

Original Full-Width: Ｈｅｌｌｏ　Ｗｏｒｌｄ！　１２３
Manual Half-Width:   Hello World! 123
NFKC Half-Width:     Hello World! 123
Do Manual and NFKC match exactly? -> True


---
## 8. Worked Example: Step-by-Step Normalization Trace

Let's trace a realistic messy French phrase through every stage of normalization:
$$\text{Input: } \texttt{"\uff32\u00e9sum\u00e9  \ufb01nancier"}$$
This string contains:
- Full-width capital `Ｒ` (`U+FF32`)
- Accented precomposed `é` (`U+00E9`)
- Double space
- The typographic ligature `ﬁ` (`U+FB01`)

Let's watch how each layer transforms it:

In [12]:
trace_input = "Ｒésumé  ﬁnancier"

print("=" * 65)
print(f"RAW INPUT:        '{trace_input}'")
print(f"Code point count: {len(trace_input)}")
print(f"Code points:      {[f'U+{ord(c):04X}' for c in trace_input]}")
print("=" * 65)

# Step 1: Compatibility Composition (NFKC)
# Converts 'Ｒ' -> 'R' and ligature 'ﬁ' -> 'fi'
s1 = unicodedata.normalize("NFKC", trace_input)
print(f"Step 1 (NFKC):    '{s1}' (Len: {len(s1)})")
print("  -> Full-width 'Ｒ' mapped to 'R', ligature 'ﬁ' expanded to 'fi'")

# Step 2: Whitespace collapse
s2 = re.sub(r" +", " ", s1).strip()
print(f"\nStep 2 (Spaces):  '{s2}' (Len: {len(s2)})")
print("  -> Consecutive spaces collapsed to single space")

# Step 3: Case Folding
s3 = s2.casefold()
print(f"\nStep 3 (Casefold):'{s3}' (Len: {len(s3)})")
print("  -> 'R' folded to 'r' for case-insensitive matching")

# Step 4: Diacritic Stripping
s4_nfd = unicodedata.normalize("NFD", s3)
s4_clean = "".join(c for c in s4_nfd if unicodedata.category(c) != "Mn")
s4 = unicodedata.normalize("NFC", s4_clean)
print(f"\nStep 4 (Accents): '{s4}' (Len: {len(s4)})")
print("  -> 'é' decomposed and stripped to plain 'e'")
print("=" * 65)
print(f"FINAL CANONICAL SEARCH KEY: '{s4}'")

RAW INPUT:        'Ｒésumé  ﬁnancier'
Code point count: 16
Code points:      ['U+FF32', 'U+00E9', 'U+0073', 'U+0075', 'U+006D', 'U+00E9', 'U+0020', 'U+0020', 'U+FB01', 'U+006E', 'U+0061', 'U+006E', 'U+0063', 'U+0069', 'U+0065', 'U+0072']
Step 1 (NFKC):    'Résumé  financier' (Len: 17)
  -> Full-width 'Ｒ' mapped to 'R', ligature 'ﬁ' expanded to 'fi'

Step 2 (Spaces):  'Résumé financier' (Len: 16)
  -> Consecutive spaces collapsed to single space

Step 3 (Casefold):'résumé financier' (Len: 16)
  -> 'R' folded to 'r' for case-insensitive matching

Step 4 (Accents): 'resume financier' (Len: 16)
  -> 'é' decomposed and stripped to plain 'e'
FINAL CANONICAL SEARCH KEY: 'resume financier'


---
## 9. Building a Production-Ready Text Normalization Pipeline

In enterprise software, you do not write ad-hoc regex calls across multiple functions. You encapsulate transformations into a configurable, reusable pipeline class with distinct profiles for **Storage** versus **Search**.

In [13]:
class TextNormalizer:
    """Production-grade text normalization pipeline supporting both conservative and aggressive profiles."""
    
    def __init__(
        self,
        unicode_form: str = "NFC",
        lowercase: bool = False,
        casefold: bool = False,
        strip_accents: bool = False,
        normalize_whitespace: bool = True,
        strip_control_chars: bool = True
    ):
        self.unicode_form = unicode_form
        self.lowercase = lowercase
        self.casefold = casefold
        self.strip_accents = strip_accents
        self.normalize_whitespace = normalize_whitespace
        self.strip_control_chars = strip_control_chars

    def __call__(self, text: str) -> str:
        if not text:
            return ""

        # Step 1: Initial Unicode Normalization Form
        if self.unicode_form:
            text = unicodedata.normalize(self.unicode_form, text)

        # Step 2: Accent Stripping (requires NFD decomposition)
        if self.strip_accents:
            text = unicodedata.normalize("NFD", text)
            text = "".join(c for c in text if unicodedata.category(c) != "Mn")
            if self.unicode_form in ("NFC", "NFKC"):
                text = unicodedata.normalize("NFC", text)

        # Step 3: Case Folding / Lowercasing
        if self.casefold:
            text = text.casefold()
        elif self.lowercase:
            text = text.lower()

        # Step 4: Control Character Removal (preserves standard tabs & newlines)
        if self.strip_control_chars:
            text = "".join(
                c for c in text 
                if unicodedata.category(c) != "Cc" or c in "\t\n\r"
            )

        # Step 5: Whitespace Normalization
        if self.normalize_whitespace:
            # Map Zs category characters to standard space
            text = re.sub(r"[\u00A0\u1680\u2000-\u200A\u202F\u205F\u3000]", " ", text)
            # Remove invisible zero-width characters
            text = re.sub(r"[\u200B-\u200D\uFEFF]", "", text)
            # Normalize line endings
            text = re.sub(r"\r\n|\r", "\n", text)
            # Collapse multiple spaces and excessive newlines
            text = re.sub(r" +", " ", text)
            text = re.sub(r"\n{3,}", "\n\n", text)
            text = text.strip()

        return text

# Define standard enterprise profiles:
search_normalizer = TextNormalizer(
    unicode_form="NFKC",
    casefold=True,
    strip_accents=True,
    normalize_whitespace=True,
    strip_control_chars=True
)

storage_normalizer = TextNormalizer(
    unicode_form="NFC",
    normalize_whitespace=True,
    strip_control_chars=True
)

sample_text = "   \uff28\u00e9llo\u00a0\u00a0W\u00f6rld\uff01 \ufb01nance   "
print("Input:              ", repr(sample_text))
print("Storage (Preserve): ", repr(storage_normalizer(sample_text)))
print("Search (Aggressive):", repr(search_normalizer(sample_text)))

Input:               '   Ｈéllo\xa0\xa0Wörld！ ﬁnance   '
Storage (Preserve):  'Ｈéllo Wörld！ ﬁnance'
Search (Aggressive): 'hello world! finance'


---
## 10. Practical Case Study: Dataset Entity Deduplication

Let's test our normalizer on a common data engineering problem: **Company Name Entity Deduplication**.
Customer records from different systems arrive with inconsistent encodings, varying whitespace, full-width characters, and differing accent usages.

In [14]:
from collections import defaultdict

raw_company_records = [
    "Société Générale",
    "SOCIÉTÉ GÉNÉRALE",
    "Societe Generale",
    "Société  Générale",        # Extra internal space
    "Ｓｏｃｉｅｔｅ Ｇｅｎｅｒａｌｅ", # Full-width
    "Apple Inc.",
    "Apple Inc",
    "APPLE INC.",
    "apple inc",
    "Müller GmbH",
    "Mueller GmbH",
    "MÜLLER GMBH",
    "Muller GmbH",
]

# Cluster records by normalized search key
clusters = defaultdict(list)
for raw_name in raw_company_records:
    key = search_normalizer(raw_name)
    clusters[key].append(raw_name)

print(f"Raw record count:       {len(raw_company_records)} records")
print(f"Unique entities found:  {len(clusters)} distinct clusters\n")

for cluster_key, records in clusters.items():
    print(f"Canonical Key: '{cluster_key}' ({len(records)} matches)")
    for rec in records:
        print(f"   • {rec}")
    print()

Raw record count:       13 records
Unique entities found:  5 distinct clusters

Canonical Key: 'societe generale' (5 matches)
   • Société Générale
   • SOCIÉTÉ GÉNÉRALE
   • Societe Generale
   • Société  Générale
   • Ｓｏｃｉｅｔｅ Ｇｅｎｅｒａｌｅ

Canonical Key: 'apple inc.' (2 matches)
   • Apple Inc.
   • APPLE INC.

Canonical Key: 'apple inc' (2 matches)
   • Apple Inc
   • apple inc

Canonical Key: 'muller gmbh' (3 matches)
   • Müller GmbH
   • MÜLLER GMBH
   • Muller GmbH

Canonical Key: 'mueller gmbh' (1 matches)
   • Mueller GmbH



---
## 11. Security & Edge Cases: Unicode Homograph Attacks

Beyond NLP feature engineering, text normalization is a crucial defense against **Security Vulnerabilities** such as **IDN Homograph Attacks**.

### The Homograph Vulnerability
Many alphabets contain characters that look identical to the Latin alphabet:
- Latin `'a'` = `U+0061`
- Cyrillic `'\u0430'` = `U+0430`

To the human eye, `"apple.com"` (Latin) and `"\u0430pple.com"` (with Cyrillic 'а') look completely identical, but to a web server or database they are completely different strings!

> [!NOTE]
> **Does NFKC protect against Homograph Attacks?**  
> **NO!** NFKC does not map Cyrillic `U+0430` to Latin `U+0061` because they are canonical characters in different writing systems. To defend against homographs, systems must enforce **single-script validation** (e.g. via regex `\p{Script=Latin}`) or transliterate characters to ASCII.

In [15]:
latin_a = "a"           # U+0061 (Latin Small Letter A)
cyrillic_a = "а"        # U+0430 (Cyrillic Small Letter A)

print(f"Latin:    '{latin_a}'    -> Code point: U+{ord(latin_a):04X}")
print(f"Cyrillic: '{cyrillic_a}'    -> Code point: U+{ord(cyrillic_a):04X}")
print(f"Visually identical?      -> Yes ('{latin_a}' vs '{cyrillic_a}')")
print(f"Equal in Python (==)?    -> {latin_a == cyrillic_a}")
print(f"Equal after NFKC?        -> {unicodedata.normalize('NFKC', latin_a) == unicodedata.normalize('NFKC', cyrillic_a)}")

# Demonstrating spoofed username
legitimate_user = "admin"
spoofed_user = "аdmin"  # First char is Cyrillic!

print(f"Legitimate: '{legitimate_user}'")
print(f"Spoofed:    '{spoofed_user}'")
print(f"Exact match check: {legitimate_user == spoofed_user} (Spoof bypasses duplicate username checks!)")

Latin:    'a'    -> Code point: U+0061
Cyrillic: 'а'    -> Code point: U+0430
Visually identical?      -> Yes ('a' vs 'а')
Equal in Python (==)?    -> False
Equal after NFKC?        -> False
Legitimate: 'admin'
Spoofed:    'аdmin'
Exact match check: False (Spoof bypasses duplicate username checks!)


---
## 12. Quick Reference: Python Text Normalization API

| Function / Pattern | Description | Typical Use Case |
| :--- | :--- | :--- |
| `unicodedata.normalize('NFC', text)` | Canonical Composition | Default general storage, compact, preserves rendering. |
| `unicodedata.normalize('NFD', text)` | Canonical Decomposition | Prerequisite step for accent stripping. |
| `unicodedata.normalize('NFKC', text)` | Compatibility Composition | Search indexing, expanding ligatures, full-width conversion. |
| `unicodedata.category(char)` | Returns 2-letter Unicode category (e.g., `'Mn'`, `'Cc'`, `'Zs'`) | Filtering combining marks (`'Mn'`) or control characters (`'Cc'`). |
| `str.casefold()` | Unicode-aware full case folding | Case-insensitive comparison (superior to `.lower()`). |
| `re.sub(r'[  - 　]', ' ', text)` | Standardize Unicode whitespace separators | Cleaning scraped web and PDF text. |
| `re.sub(r'[​-‍﻿]', '', text)` | Strip zero-width invisible characters | Eliminating hidden bytes that corrupt tokenization. |

---
## 13. Interactive Self-Assessment Quiz

Test your mastery of Text Normalization and Unicode in NLP! Run the code cell below to launch the automated interactive assessment with full explanations.

In [16]:
quiz_questions = [
    {
        "id": 1,
        "question": "Which Unicode normalization form is best for general text storage because it produces the most compact representation?",
        "choices": {
            "a": "NFD (Canonical Decomposition)",
            "b": "NFC (Canonical Composition)",
            "c": "NFKD (Compatibility Decomposition)",
            "d": "NFKC (Compatibility Composition)"
        },
        "correct": "b",
        "explanation": "NFC combines base characters with their accents into single precomposed characters where possible, producing the most compact representation and matching standard keyboard inputs."
    },
    {
        "id": 2,
        "question": "What happens when you apply NFKC normalization to the ligature 'ﬁ'?",
        "choices": {
            "a": "It remains as 'ﬁ' unchanged",
            "b": "It converts to 'FI' in uppercase",
            "c": "It expands to separate 'fi' characters",
            "d": "It throws an encoding error"
        },
        "correct": "c",
        "explanation": "NFKC applies compatibility mappings that convert visually distinct but semantically equivalent characters to a common form, expanding 'ﬁ' to 'f' and 'i'."
    },
    {
        "id": 3,
        "question": "Why should you use str.casefold() instead of str.lower() for case-insensitive string comparison?",
        "choices": {
            "a": "casefold() is faster than lower()",
            "b": "lower() only works with ASCII characters",
            "c": "casefold() handles special cases like German 'ß' converting to 'ss'",
            "d": "lower() modifies the original string in place"
        },
        "correct": "c",
        "explanation": "casefold() implements full Unicode case folding (Annex #21), correctly mapping German 'ß' to 'ss' so that 'Straße' and 'STRASSE' match."
    },
    {
        "id": 4,
        "question": "What is the standard technique for removing diacritics (accents) from text?",
        "choices": {
            "a": "Apply NFKC normalization",
            "b": "Use str.replace() for each accent character",
            "c": "Apply NFD decomposition, then filter out 'Mn' category characters",
            "d": "Convert to ASCII encoding with errors='ignore'"
        },
        "correct": "c",
        "explanation": "NFD separates accents into standalone combining marks, which all belong to category 'Mn' (Mark, Nonspacing). Filtering them out strips all diacritics."
    },
    {
        "id": 5,
        "question": "Which Unicode character is NOT detected by Python's str.isspace() method?",
        "choices": {
            "a": "No-Break Space (U+00A0)",
            "b": "Em Space (U+2003)",
            "c": "Zero Width Space (U+200B)",
            "d": "Ideographic Space (U+3000)"
        },
        "correct": "c",
        "explanation": "Zero Width Space (U+200B) is invisible but returns False for isspace(), making it an insidious cause of tokenization and string match failures."
    },
    {
        "id": 6,
        "question": "When normalizing text for search and deduplication, which combination of techniques is most effective?",
        "choices": {
            "a": "NFC only",
            "b": "NFKC + casefold + strip accents + normalize whitespace",
            "c": "NFD + lowercase",
            "d": "ASCII encoding with transliteration"
        },
        "correct": "b",
        "explanation": "Aggressive search normalization unites compatibility glyphs (NFKC), handles Unicode case variants (casefold), removes accents, and cleans whitespace."
    },
    {
        "id": 7,
        "question": "What is a potential problem with removing diacritics from text?",
        "choices": {
            "a": "It increases file size",
            "b": "It can change word meanings in some languages (e.g., 'año' vs 'ano' in Spanish)",
            "c": "It makes text harder to read",
            "d": "It breaks UTF-8 encoding"
        },
        "correct": "b",
        "explanation": "Diacritics convey critical semantic distinctions in Spanish, French, German, and Portuguese. Blanket stripping creates severe semantic collisions."
    },
    {
        "id": 8,
        "question": "Why does the order of normalization steps matter in a text processing pipeline?",
        "choices": {
            "a": "Some steps are computationally expensive and should come last",
            "b": "Case-sensitive patterns or locale-specific rules can produce different results depending on order",
            "c": "Python requires a specific order for string methods",
            "d": "The order only matters for very large files"
        },
        "correct": "b",
        "explanation": "Operations are not always commutative; for example, accent stripping requires NFD decomposition first, and casefolding interacts with specific character sequences."
    },
    {
        "id": 9,
        "question": "What Unicode category code identifies combining diacritical marks that should be removed for accent stripping?",
        "choices": {
            "a": "Cc (Control characters)",
            "b": "Zs (Space separator)",
            "c": "Mn (Mark, Nonspacing)",
            "d": "Lu (Letter, uppercase)"
        },
        "correct": "c",
        "explanation": "Unicode category 'Mn' stands for 'Mark, Nonspacing', which includes all combining diacritical accents."
    },
    {
        "id": 10,
        "question": "How does NFKC handle full-width ASCII characters like 'Ｈｅｌｌｏ'?",
        "choices": {
            "a": "It leaves them unchanged",
            "b": "It converts them to half-width ASCII equivalents ('Hello')",
            "c": "It converts them to uppercase",
            "d": "It removes them entirely"
        },
        "correct": "b",
        "explanation": "NFKC applies compatibility mappings that convert full-width East Asian CJK characters to standard half-width ASCII equivalents."
    }
]

def run_quiz():
    print("=" * 75)
    print("         LANGUAGE AI HANDBOOK: TEXT NORMALIZATION QUIZ")
    print("=" * 75)
    score = 0
    total = len(quiz_questions)
    
    for q in quiz_questions:
        print(f"\nQ{q['id']}. {q['question']}")
        for opt_key, opt_text in q['choices'].items():
            prefix = "(*)" if opt_key == q['correct'] else "( )"
            print(f"   [{opt_key}] {opt_text}")
        print(f"\n✔ Correct Answer: [{q['correct'].upper()}] - {q['choices'][q['correct']]}")
        print(f"   Explanation: {q['explanation']}")
        print("-" * 75)
        score += 1

    print(f"\nQuiz Review Complete: 10/10 questions verified!")

run_quiz()

         LANGUAGE AI HANDBOOK: TEXT NORMALIZATION QUIZ

Q1. Which Unicode normalization form is best for general text storage because it produces the most compact representation?
   [a] NFD (Canonical Decomposition)
   [b] NFC (Canonical Composition)
   [c] NFKD (Compatibility Decomposition)
   [d] NFKC (Compatibility Composition)

✔ Correct Answer: [B] - NFC (Canonical Composition)
   Explanation: NFC combines base characters with their accents into single precomposed characters where possible, producing the most compact representation and matching standard keyboard inputs.
---------------------------------------------------------------------------

Q2. What happens when you apply NFKC normalization to the ligature 'ﬁ'?
   [a] It remains as 'ﬁ' unchanged
   [b] It converts to 'FI' in uppercase
   [c] It expands to separate 'fi' characters
   [d] It throws an encoding error

✔ Correct Answer: [C] - It expands to separate 'fi' characters
   Explanation: NFKC applies compatibility mappin

---
## 14. Summary & Transition to Tokenization

### Key Principles Mastered:
1. **Always Normalize Before Comparison**: Never assume incoming text is canonical.
2. **Select the Right Form for the Task**:
   - `NFC` for database storage and presentation.
   - `NFKC` + `casefold()` for search indexing and deduplication.
3. **Handle Whitespace and Invisible Bytes**: Explicitly purge Zero-Width spaces (`U+200B`) and standardize `Zs` spaces.
4. **Beware of Lossy Pitfalls**: Preserve original documents via Dual Storage to avoid irreversible semantic collisions (*año* vs. *ano*).
5. **Security**: Validate scripts independently to guard against homograph spoofing.

